In [7]:
import json
import joblib
import pandas as pd
from src.config import INTERIM_DIR, MODELS_DIR

# Carrega o dataset com scores
df = pd.read_parquet(INTERIM_DIR / "dataset_analitico_com_scores.parquet")

# Carrega o modelo (caminho absoluto via config)
modelo = joblib.load(MODELS_DIR / "modelo_supervisionado_anomalia.joblib")

# Quais features o modelo usa?
if hasattr(modelo, "feature_names_in_"):
    print("Features que o modelo usa:")
    for f in modelo.feature_names_in_:
        print(f"  - {f}")

# Feature importance
if hasattr(modelo, "feature_importances_"):
    import numpy as np
    importances = pd.Series(
        modelo.feature_importances_,
        index=modelo.feature_names_in_
    ).sort_values(ascending=False)
    print("\nFeature importances (top 15):")
    print(importances.head(15))
    # E ver as 3 features mais importantes:
    print("\nTop 3 features mais importantes:")
    print(importances.head(3))


# Segurança ao checar nomes nas features
feature_names = getattr(modelo, "feature_names_in_", [])
print("anomalia_score está nas features?", 
      "anomalia_score" in feature_names)
print("anomalia_label está nas features?",
      "anomalia_label" in feature_names)

Features que o modelo usa:
  - valor_licitacao
  - n_participantes
  - n_itens
  - valor_total_itens
  - hhi
  - top1_share
  - log_valor_licitacao
  - ano
  - dia_semana
  - modalidade_compra
  - uf
  - situacao_licitacao
  - ano_mes_arquivo
anomalia_score está nas features? False
anomalia_label está nas features? False


In [2]:
import joblib
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from src.config import INTERIM_DIR, MODELS_DIR

# ============================================================
# DIAGNÓSTICO 1: Entender a estrutura do Pipeline
# ============================================================
modelo = joblib.load(MODELS_DIR / "modelo_supervisionado_anomalia.joblib")

print("Tipo do objeto:", type(modelo).__name__)
print("\nEtapas do Pipeline:")
for nome, etapa in modelo.named_steps.items():
    print(f"  - {nome}: {type(etapa).__name__}")

# Tenta achar o classificador final
classificador = None
for nome, etapa in modelo.named_steps.items():
    if hasattr(etapa, "feature_importances_"):
        classificador = etapa
        nome_clf = nome
        break

if classificador is None:
    print("\n⚠️ Nenhuma etapa tem feature_importances_")
    print("Etapa final:", list(modelo.named_steps.values())[-1])
else:
    print(f"\n✓ Classificador encontrado na etapa '{nome_clf}'")
    print(f"  Tipo: {type(classificador).__name__}")
    print(f"  Número de features (após transform): {len(classificador.feature_importances_)}")
    
    # Tenta recuperar nomes das features após o transform
    try:
        feature_names = modelo[:-1].get_feature_names_out()
        importances = pd.Series(
            classificador.feature_importances_,
            index=feature_names
        ).sort_values(ascending=False)
        
        print("\n" + "=" * 60)
        print("FEATURE IMPORTANCES (após preprocessing)")
        print("=" * 60)
        print(importances.head(20).round(4))
        print(f"\nTop 3 somam: {importances.head(3).sum():.2%}")
        print(f"Top 5 somam: {importances.head(5).sum():.2%}")
    except Exception as e:
        print(f"\n⚠️ Erro ao recuperar nomes: {e}")
        print("\nImportâncias (sem nomes):")
        print(classificador.feature_importances_[:20])

# ============================================================
# DIAGNÓSTICO 2: Label honesta
# ============================================================
df = pd.read_parquet(INTERIM_DIR / "dataset_analitico_com_scores.parquet")

print("\n" + "=" * 60)
print("LABEL HONESTA PROPOSTA")
print("=" * 60)

criterios = [c for c in df.columns if c.startswith("criterio_")]
print(f"Critérios disponíveis: {criterios}")

if "criterio_2_sem_competicao" in df.columns:
    label_honesta = (
        df["criterio_2_sem_competicao"].fillna(False).astype(bool) |
        df["criterio_3_limite_dispensa"].fillna(False).astype(bool) |
        df["criterio_4_alto_valor_sem_comp"].fillna(False).astype(bool)
    ).astype(int)
    
    print(f"\nDistribuição:")
    print(label_honesta.value_counts())
    print(f"Taxa de positivos: {label_honesta.mean()*100:.2f}%")

# ============================================================
# DIAGNÓSTICO 3: Anos disponíveis
# ============================================================
print("\n" + "=" * 60)
print("VOLUMETRIA POR ANO")
print("=" * 60)
if "ano" in df.columns:
    print(df["ano"].value_counts().sort_index())
else:
    print("Coluna 'ano' ausente")
    if "ano_mes_arquivo" in df.columns:
        anos = df["ano_mes_arquivo"].astype(str).str[:4]
        print(anos.value_counts().sort_index())

Tipo do objeto: Pipeline

Etapas do Pipeline:
  - preprocess: ColumnTransformer
  - model: RandomForestClassifier

✓ Classificador encontrado na etapa 'model'
  Tipo: RandomForestClassifier
  Número de features (após transform): 82

FEATURE IMPORTANCES (após preprocessing)
valor_total_itens                                    0.1723
n_itens                                              0.1610
n_participantes                                      0.1354
top1_share                                           0.0837
valor_licitacao                                      0.0824
log_valor_licitacao                                  0.0817
hhi                                                  0.0771
situacao_licitacao_Encerrado                         0.0554
modalidade_compra_Pregão - Registro de Preço         0.0481
modalidade_compra_Dispensa de Licitação              0.0416
modalidade_compra_Pregão                             0.0111
situacao_licitacao_Publicado                         0.0095
modali

In [3]:
import json
from pathlib import Path

import joblib
import pandas as pd

from src.config import INTERIM_DIR, MODELS_DIR

# 1. Métricas atuais
with open(MODELS_DIR / 'metricas_supervisionado_anomalia.json', encoding='utf-8') as arquivo_metricas:
    metricas = json.load(arquivo_metricas)
print('=== MÉTRICAS ATUAIS ===')
print(json.dumps(metricas, indent=2, ensure_ascii=False))

# 2. Volumetria dos dados
print('\n=== VOLUMETRIA ===')
df = pd.read_parquet(INTERIM_DIR / 'dataset_analitico.parquet')
print(f'Total: {len(df):,}')
if 'ano' in df.columns:
    print(df['ano'].value_counts().sort_index())

# 3. Modelo salvo - quais features usa agora?
print('\n=== FEATURES DO MODELO ===')
modelo = joblib.load(MODELS_DIR / 'modelo_supervisionado_anomalia.joblib')
if hasattr(modelo, 'named_steps'):
    for nome, etapa in modelo.named_steps.items():
        print(f"Etapa '{nome}': {type(etapa).__name__}")
    try:
        feature_names = modelo[:-1].get_feature_names_out()
        print(f'\nTotal de features após preprocessing: {len(feature_names)}')
        print('Primeiras 10:', feature_names[:10].tolist())
    except Exception as erro:
        print(f'Erro: {erro}')

=== MÉTRICAS ATUAIS ===
{
  "accuracy": 0.9944673068129858,
  "precision": 0.969639468690702,
  "recall": 0.9198919891989199,
  "f1": 0.9441108545034642,
  "confusion_matrix": [
    [
      20727,
      32
    ],
    [
      89,
      1022
    ]
  ],
  "roc_auc": 0.9995531852428945,
  "pr_auc": 0.9917566847548234
}

=== VOLUMETRIA ===
Total: 109,346
ano
2022    109346
Name: count, dtype: int64

=== FEATURES DO MODELO ===
Etapa 'pre': ColumnTransformer
Etapa 'clf': RandomForestClassifier

Total de features após preprocessing: 82
Primeiras 10: ['valor_licitacao', 'n_participantes', 'n_itens', 'valor_total_itens', 'hhi', 'top1_share', 'log_valor_licitacao', 'ano', 'dia_semana', 'modalidade_compra_Concorrência']


In [4]:
import pandas as pd
from src.config import INTERIM_DIR

df = pd.read_parquet(INTERIM_DIR / "dataset_analitico_com_scores.parquet")

print("=" * 60)
print("DISTRIBUIÇÃO DE CADA CRITÉRIO INDIVIDUALMENTE")
print("=" * 60)

for col in ["criterio_1_acima_p99_modalidade",
            "criterio_2_sem_competicao",
            "criterio_3_limite_dispensa",
            "criterio_4_alto_valor_sem_comp"]:
    if col in df.columns:
        positivos = df[col].sum()
        total = len(df)
        print(f"\n{col}:")
        print(f"  Positivos: {positivos:,} ({positivos/total*100:.2f}%)")

print("\n" + "=" * 60)
print("COMBINAÇÕES PARA TESTAR")
print("=" * 60)

# Opção A: só critério 3 + 4 (sem competição já está em criterio_2)
opcao_a = (
    df["criterio_3_limite_dispensa"].fillna(False).astype(bool) |
    df["criterio_4_alto_valor_sem_comp"].fillna(False).astype(bool)
).astype(int)
print(f"\nOpção A — criterio_3 OR criterio_4:")
print(f"  Positivos: {opcao_a.sum():,} ({opcao_a.mean()*100:.2f}%)")

# Opção B: TODOS combinados
opcao_b = (
    df["criterio_1_acima_p99_modalidade"].fillna(False).astype(bool) |
    df["criterio_2_sem_competicao"].fillna(False).astype(bool) |
    df["criterio_3_limite_dispensa"].fillna(False).astype(bool) |
    df["criterio_4_alto_valor_sem_comp"].fillna(False).astype(bool)
).astype(int)
print(f"\nOpção B — TODOS critérios:")
print(f"  Positivos: {opcao_b.sum():,} ({opcao_b.mean()*100:.2f}%)")

# Opção C: pelo menos 2 critérios (mais restritiva)
opcao_c = (
    df["criterio_1_acima_p99_modalidade"].fillna(False).astype(int) +
    df["criterio_2_sem_competicao"].fillna(False).astype(int) +
    df["criterio_3_limite_dispensa"].fillna(False).astype(int) +
    df["criterio_4_alto_valor_sem_comp"].fillna(False).astype(int)
) >= 2
print(f"\nOpção C — pelo menos 2 critérios:")
print(f"  Positivos: {opcao_c.sum():,} ({opcao_c.mean()*100:.2f}%)")

DISTRIBUIÇÃO DE CADA CRITÉRIO INDIVIDUALMENTE

criterio_1_acima_p99_modalidade:
  Positivos: 1,100 (1.01%)

criterio_2_sem_competicao:
  Positivos: 69,421 (63.49%)

criterio_3_limite_dispensa:
  Positivos: 779 (0.71%)

criterio_4_alto_valor_sem_comp:
  Positivos: 391 (0.36%)

COMBINAÇÕES PARA TESTAR

Opção A — criterio_3 OR criterio_4:
  Positivos: 1,170 (1.07%)

Opção B — TODOS critérios:
  Positivos: 69,849 (63.88%)

Opção C — pelo menos 2 critérios:
  Positivos: 1,501 (1.37%)


In [5]:
print('anomalia_score está nas features?', 'anomalia_score' in modelo.feature_names_in_)
print('anomalia_label está nas features?', 'anomalia_label' in modelo.feature_names_in_)

# E ver as 3 features mais importantes:
print('\nTop 3 features mais importantes:')
print(importances.head(3))

anomalia_score está nas features? False
anomalia_label está nas features? False

Top 3 features mais importantes:
valor_total_itens    0.172314
n_itens              0.161048
n_participantes      0.135373
dtype: float64
